In [2]:
# -----------------------------------------------------------
# 0. 환경설정 & 데이터 로드
# -----------------------------------------------------------
import pandas as pd
pd.set_option('display.max_columns', None)   # 컬럼 다 보이게
pd.set_option('display.width', 150)          # 줄바꿈 없이 넓게 출력

DATA_PATH = "../data/co2_emissions_long.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("데이터 로드 완료")
print("shape:", df.shape)
print(df.head(8))

데이터 로드 완료
shape: (1320, 7)
        Code            Name  Year                   Sector  Emissions_kt  Population_000s  PerCapita_Emissions_t
0  E09000001  City of London  2005                 Domestic     20.346117            7.131             228.765253
1  E09000001  City of London  2005              Grand total   1631.325022            7.131             228.765253
2  E09000001  City of London  2005  Industry and Commercial   1545.660538            7.131             228.765253
3  E09000001  City of London  2005                Transport     65.318367            7.131             228.765253
4  E09000001  City of London  2006                 Domestic     20.397841            7.254             243.338377
5  E09000001  City of London  2006              Grand total   1765.176589            7.254             243.338377
6  E09000001  City of London  2006  Industry and Commercial   1679.655916            7.254             243.338377
7  E09000001  City of London  2006                Transport  

In [5]:
# -----------------------------------------------------------
# 1. 결측치 · 타입 · 이상치 점검
# -----------------------------------------------------------
print("\n" + "=" * 60)
print("[1-1] 데이터 타입 확인")
print(df.dtypes)

print("\n" + "=" * 60)
print("[1-2] 결측치 확인")
na_count = df.isna().sum()
print(na_count)

print("\n" + "=" * 60)
print("[1-3] Sector 값 종류 확인 (Grand total이 섞여있는지 확인용)")
print(df['Sector'].value_counts())

print("\n" + "=" * 60)
print("[1-4] 자치구(Name) 개수 & 연도 범위 확인")
print("자치구 수:", df['Name'].nunique())
print("연도 범위:", df['Year'].min(), "~", df['Year'].max())

print("\n" + "=" * 60)
print("[1-5] 이상치 후보 확인 - City of London")
# Grand total 기준으로 자치구별 평균 배출량을 비교해서
# City of London이 다른 자치구 대비 몇 배나 높은지 확인
grand = df[df['Sector'] == 'Grand total']
avg_by_borough = grand.groupby('Name')['Emissions_kt'].mean().sort_values(ascending=False)
print(avg_by_borough.head(5))

col_avg = avg_by_borough['City of London']
rest_avg = avg_by_borough.drop('City of London').mean()
print(f"\nCity of London 평균 배출량: {col_avg:,.1f} kt")
print(f"나머지 32개 자치구 평균: {rest_avg:,.1f} kt")
print(f"City of London은 나머지 평균의 약 {col_avg / rest_avg:.1f}배")


[1-1] 데이터 타입 확인
Code                         str
Name                         str
Year                       int64
Sector                       str
Emissions_kt             float64
Population_000s          float64
PerCapita_Emissions_t    float64
dtype: object

[1-2] 결측치 확인
Code                     0
Name                     0
Year                     0
Sector                   0
Emissions_kt             0
Population_000s          0
PerCapita_Emissions_t    0
dtype: int64

[1-3] Sector 값 종류 확인 (Grand total이 섞여있는지 확인용)
Sector
Domestic                   330
Grand total                330
Industry and Commercial    330
Transport                  330
Name: count, dtype: int64

[1-4] 자치구(Name) 개수 & 연도 범위 확인
자치구 수: 33
연도 범위: 2005 ~ 2014

[1-5] 이상치 후보 확인 - City of London
Name
Westminster      3184.180892
Tower Hamlets    2108.574459
Hillingdon       1956.089723
Camden           1601.008784
Ealing           1583.198704
Name: Emissions_kt, dtype: float64

City of London 평균 배출량: 1,522.3 kt
나머지 

In [6]:
# -----------------------------------------------------------
# 2. 기술통계 확인
# -----------------------------------------------------------
print("\n" + "=" * 60)
print("[2-1] 부문(Sector)별 기술통계")
print(df.groupby('Sector')['Emissions_kt'].describe().round(2))

print("\n" + "=" * 60)
print("[2-2] 연도별 런던 전체(Grand total 합계) 배출량 추이")
london_total_by_year = grand.groupby('Year')['Emissions_kt'].sum()
print(london_total_by_year)

print("\n" + "=" * 60)
print("[2-3] 부문별 연도별 평균 배출량 (부문 비중 변화 확인용)")
sector_year = df[df['Sector'] != 'Grand total'].groupby(['Year', 'Sector'])['Emissions_kt'].sum().unstack()
print(sector_year.round(1))
# 부문별 비중(%) 도 함께 계산
sector_year_pct = sector_year.div(sector_year.sum(axis=1), axis=0) * 100
print("\n부문별 비중(%)")
print(sector_year_pct.round(1))


[2-1] 부문(Sector)별 기술통계
                         count     mean     std     min     25%      50%      75%      max
Sector                                                                                    
Domestic                 330.0   474.92  148.87   16.51  389.55   465.92   551.20   870.70
Grand total              330.0  1296.98  484.38  604.52  956.23  1205.87  1511.22  3564.86
Industry and Commercial  330.0   583.44  459.30  182.67  294.06   432.59   674.24  2712.20
Transport                330.0   238.63   83.51   50.46  172.29   239.22   288.65   544.27

[2-2] 연도별 런던 전체(Grand total 합계) 배출량 추이
Year
2005    46158.631190
2006    47306.561600
2007    45997.242819
2008    45952.154714
2009    41636.511730
2010    43861.256812
2011    39298.474207
2012    42157.600461
2013    40531.028372
2014    35103.232537
Name: Emissions_kt, dtype: float64

[2-3] 부문별 연도별 평균 배출량 (부문 비중 변화 확인용)
Sector  Domestic  Industry and Commercial  Transport
Year                                              